# Verificación del Entorno y GPU asignada

In [ ]:
import torch
import subprocess

# Verificar disponibilidad de CUDA
cuda_available = torch.cuda.is_available()
print(f"GPU disponible: {cuda_available}")

if cuda_available:
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    # Mostrar detalles de uso de memoria y procesos
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Advertencia: No se detectó GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución y selecciona T4 GPU.")

GPU disponible: True
Dispositivo: Tesla T4
Wed Oct  7 01:37:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----

# Instalación de dependencias
Instalamos las librerías necesarias del ecosistema de Hugging Face y herramientas para métricas.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn torch huggingface_hub accelerate pandas
print("Dependencias instaladas correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
Dependencias instaladas correctamente.


# Autenticación en Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

# Carga de modelos y datasets

In [ ]:
from datasets import load_dataset

print("Descargando dataset SST-2...")
dataset = load_dataset("stanfordnlp/sst2", split="validation")

# Extraemos textos y etiquetas reales para evaluación
texts = dataset["sentence"]
true_labels = dataset["label"]

print(f"Dataset cargado. Total de ejemplos a evaluar: {len(texts)}")
print(f"Ejemplo: '{texts[0]}' -> Etiqueta: {true_labels[0]}")

Descargando dataset SST-2...


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset cargado. Total de ejemplos a evaluar: 872
Ejemplo: 'it 's a charming and often affecting journey . ' -> Etiqueta: 1


# Configuración y Pipeline de Inferencia
Implementamos un bucle que carga cada modelo, procesa el dataset por lotes (batches) para optimizar la GPU, calcula las predicciones, mide la latencia y extrae las métricas.

In [ ]:
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Definimos los modelos a comparar
model_ids = [
    "distilbert-base-uncased-finetuned-sst-2-english", # Modelo destilado (rápido)
    "textattack/bert-base-uncased-SST-2",              # BERT estándar
    "textattack/roberta-base-SST-2"                    # RoBERTa (generalmente más robusto)
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
resultados = []

for model_id in model_ids:
    nombre_corto = model_id.split("/")[-1]
    print(f"\n--- Iniciando evaluación de: {nombre_corto} ---")

    # 1. Carga de Modelo y Tokenizador
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device)
    model.eval()

    predictions = []

    # 2. Pipeline de inferencia y medición de latencia
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start_time = time.time()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    end_time = time.time()

    # 3. Cálculo de Latencia
    latencia_total = end_time - start_time
    latencia_por_ejemplo = (latencia_total / len(texts)) * 1000 # en milisegundos

    # 4. Cálculo de Métricas (accuracy, precision, recall, f1)
    acc = accuracy_score(true_labels, predictions)
    prec = precision_score(true_labels, predictions, average='binary')
    rec = recall_score(true_labels, predictions, average='binary')
    f1 = f1_score(true_labels, predictions, average='binary')

    resultados.append({
        "Modelo": nombre_corto,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Latencia Total (s)": round(latencia_total, 3),
        "Latencia/Ejemplo (ms)": round(latencia_por_ejemplo, 2)
    })

    # Liberar memoria de GPU
    del model, tokenizer, inputs, outputs
    torch.cuda.empty_cache()


--- Iniciando evaluación de: distilbert-base-uncased-finetuned-sst-2-english ---


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-SST-2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Tablas y Resúmenes
Consolidamos los resultados en un DataFrame para su fácil visualización e interpretación.

In [ ]:

from IPython.display import display

# Crear y mostrar la tabla de resultados
df_resultados = pd.DataFrame(resultados)
df_resultados.set_index("Modelo", inplace=True)

print("=== Resumen de Métricas y Rendimiento ===")
display(df_resultados)

=== Resumen de Métricas y Rendimiento ===


,Accuracy,Precision,Recall,F1-Score,Latencia Total (s),Latencia/Ejemplo (ms)
Modelo,,,,,,
distilbert-base-uncased-finetuned-sst-2-english,0.9106,0.8978,0.9302,0.9137,2.394,2.75
bert-base-uncased-SST-2,0.9243,0.9181,0.9347,0.9263,2.157,2.47
roberta-base-SST-2,0.9404,0.9375,0.9459,0.9417,2.719,3.12


# Scripts Reproducibles (Obtención y Exportación)

In [1]:
# 1. Instalación de dependencias (si se ejecuta en un entorno nuevo)
# pip install datasets pandas

import pandas as pd
from datasets import load_dataset

def preparar_dataset_sst2(export_path="sst2_validation_sample.csv"):
    """
    Descarga el dataset SST-2, imprime información de su estructura
    y lo exporta a un archivo CSV.
    """
    print("Iniciando la descarga del dataset SST-2 (versión Stanford)...")

    # Cargamos el split de validación para evaluación rápida
    # Usamos el namespace de stanfordnlp para evitar errores de URI obsoletos
    dataset = load_dataset("stanfordnlp/sst2", split="validation")

    print("\n--- Estructura del Dataset ---")
    print(dataset)
    print("\nCaracterísticas (Columnas):")
    print(dataset.features)

    # Convertimos a un DataFrame de pandas para facilitar la manipulación y exportación
    df = dataset.to_pandas()

    # Mapear las etiquetas numéricas a texto para mayor claridad humana
    mapa_etiquetas = {0: "Negativo", 1: "Positivo"}
    df['label_text'] = df['label'].map(mapa_etiquetas)

    print("\n--- Muestra de los primeros 5 registros ---")
    print(df[['idx', 'sentence', 'label', 'label_text']].head())

    # Distribución de clases para verificar balance
    print("\n--- Distribución de Clases ---")
    print(df['label_text'].value_counts())

    # Exportar a CSV
    df.to_csv(export_path, index=False, encoding='utf-8')
    print(f"\nDataset exportado exitosamente a: {export_path}")

if __name__ == "__main__":
    preparar_dataset_sst2()

Iniciando la descarga del dataset SST-2 (versión Stanford)...


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]


--- Estructura del Dataset ---
Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 872
})

Características (Columnas):
{'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}

--- Muestra de los primeros 5 registros ---
   idx                                           sentence  label label_text
0    0    it 's a charming and often affecting journey .       1   Positivo
1    1                 unflinchingly bleak and desperate       0   Negativo
2    2  allows us to hope that nolan is poised to emba...      1   Positivo
3    3  the acting , costumes , music , cinematography...      1   Positivo
4    4                  it 's slow -- very , very slow .       0   Negativo

--- Distribución de Clases ---
label_text
Positivo    444
Negativo    428
Name: count, dtype: int64

Dataset exportado exitosamente a: sst2_validation_sample.csv
